[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jiehou-lab/urban-ai/blob/main/notebooks/lab4_classification.ipynb)

# Lab 4: Classification

**Duration:** ~1.0 hour
**TA lead:** Sean
**Course:** Urban AI — AI-Driven Decision Support for Real-World Urban Challenges (MSU AI-Ready Initiative)

## Learning goals
- Train a simple classifier to predict building condition from parcel features.
- Read a confusion matrix and compute overall accuracy.
- Compute error rates by subgroup and recognize when a model is less reliable for some neighborhoods than others.
- Produce an error table by subgroup with a fairness note.


## Before you start: Track A vs. Track B

Every Urban AI lab has two tracks. Both produce the **same artifact**: **Error table by subgroup + fairness note**.

- **Track A — No code (default).** Use a no-code classifier UI to label examples, train a model, and inspect its errors. No installation, no Python required — use this track if you would rather click through a web tool.
- **Track B — Colab (this notebook).** Train logistic regression / a decision tree on parcel data; inspect the confusion matrix and subgroup error rates. You will run pre-written cells and change only the parameters marked `# ▶ CHANGE ME`. You will never need to write code from scratch.

Both tracks end with the same 4 reflection prompts (the last cell of this notebook).


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score

RNG = np.random.default_rng(21)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
print("Setup complete.")

## 2. Load the data: parcel / building-condition labels

> **Synthetic-but-realistic data.** The dataset below is generated in this notebook with a fixed
> random seed so the lab runs the same way for everyone, completely offline. It is built to look and
> behave like real urban data, but it is not real. To swap in real data for your own city, instructors
> can replace the data-generation cell with a download/load from a real source such as:
- City/county property assessor parcel datasets (often on the county GIS or open data portal)
- Code enforcement violation records, published by many city open data portals
- HUD or local housing-condition survey data


In [ ]:
n = 800
neighborhood_tier = RNG.choice(["High-Income", "Mixed-Income", "Low-Income"], size=n, p=[0.3, 0.4, 0.3])

year_built = RNG.integers(1900, 2020, size=n)
sqft = RNG.normal(1500, 400, size=n).clip(400, None)

tier_maintenance_mean = {"High-Income": 75, "Mixed-Income": 60, "Low-Income": 48}
maintenance_score = np.array(
    [RNG.normal(tier_maintenance_mean[t], 12) for t in neighborhood_tier]
).clip(0, 100)
distance_to_transit_km = RNG.exponential(1.5, size=n).clip(0.05, 10)

# The TRUE underlying condition depends on age and maintenance.
age = 2025 - year_built
poor_prob = 1 / (1 + np.exp(-(1.2 + 0.025 * age - 0.075 * maintenance_score)))
true_poor = RNG.random(n) < poor_prob

# LABEL NOISE: inspection/reporting is noisier and sparser in under-resourced Low-Income tracts
# (a common real-world pattern -- fewer inspectors, less consistent code enforcement), so the
# *recorded* labels we train on are less reliable there than the true condition would suggest.
label_noise_rate = np.where(neighborhood_tier == "Low-Income", 0.28,
                    np.where(neighborhood_tier == "Mixed-Income", 0.12, 0.05))
flip = RNG.random(n) < label_noise_rate
observed_poor = np.where(flip, ~true_poor, true_poor)

parcels = pd.DataFrame({
    "parcel_id": [f"P{10000+i}" for i in range(n)],
    "neighborhood_tier": neighborhood_tier,
    "year_built": year_built,
    "sqft": sqft.round(0),
    "maintenance_score": maintenance_score.round(1),
    "distance_to_transit_km": distance_to_transit_km.round(2),
    "condition_label": np.where(observed_poor, "Poor", "Good"),
})
parcels.head()

### Split into training and test sets
We hold out 30% of parcels the model never sees during training, so we can honestly measure how well it generalizes -- and, crucially, whether it generalizes equally well across neighborhoods.

In [ ]:
FEATURES = ["year_built", "sqft", "maintenance_score", "distance_to_transit_km"]
X = parcels[FEATURES]
y = (parcels["condition_label"] == "Poor").astype(int)

X_train, X_test, y_train, y_test, tier_train, tier_test = train_test_split(
    X, y, parcels["neighborhood_tier"], test_size=0.3, random_state=42, stratify=y
)
print(f"Training rows: {len(X_train)}, Test rows: {len(X_test)}")

### Train a classifier
Logistic regression is a simple, interpretable baseline -- a reasonable first model for a planning department to pilot before considering anything more complex.

In [ ]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
overall_acc = accuracy_score(y_test, y_pred)
print(f"Overall test accuracy: {overall_acc:.2%}")

### Read the confusion matrix
Accuracy alone can hide important mistakes -- the confusion matrix shows exactly which buildings were mislabeled 'Good' when they were actually 'Poor' (and vice versa).

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig_confusion, ax = plt.subplots(figsize=(5, 5))
ax.imshow(cm, cmap="Blues")
ax.set_title("Confusion Matrix: Building Condition Classifier")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Good", "Poor"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Good", "Poor"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="black", fontsize=14)
plt.tight_layout()
plt.show()

### Responsible AI check: error rates by subgroup
An overall accuracy number can look fine while hiding much larger errors for one neighborhood tier. This is exactly the kind of gap the course's fairness policy asks you to check before using a model's output.

In [ ]:
results = X_test.copy()
results["true_label"] = y_test.values
results["pred_label"] = y_pred
results["neighborhood_tier"] = tier_test.values
results["correct"] = results["true_label"] == results["pred_label"]

subgroup_error = results.groupby("neighborhood_tier").apply(
    lambda g: pd.Series({
        "n": len(g),
        "accuracy": g["correct"].mean(),
        "error_rate": 1 - g["correct"].mean(),
        "false_positive_rate": ((g["pred_label"] == 1) & (g["true_label"] == 0)).sum() / max((g["true_label"] == 0).sum(), 1),
        "false_negative_rate": ((g["pred_label"] == 0) & (g["true_label"] == 1)).sum() / max((g["true_label"] == 1).sum(), 1),
    }),
    include_groups=False,
).round(3)
subgroup_error

In [ ]:
fig_subgroup, ax = plt.subplots(figsize=(7, 5))
subgroup_error["error_rate"].plot(kind="bar", ax=ax, color="tab:red")
ax.set_title("Classifier Error Rate by Neighborhood Tier")
ax.set_xlabel("Neighborhood tier")
ax.set_ylabel("Error rate (1 - accuracy)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Experiment

Try changing the parameters marked `# ▶ CHANGE ME` in the next cell(s) and re-run. Specifically, try:

1. Change `MODEL_TYPE` to `"tree"` and re-run -- does the subgroup error gap get better or worse?
2. Change `DECISION_THRESHOLD` to 0.3 or 0.7 -- how does that shift false positives vs. false negatives?
3. Remove `maintenance_score` from `FEATURES_EXPERIMENT` -- does accuracy drop more for one tier than another?


In [ ]:
# ▶ CHANGE ME: try "tree" instead of "logistic"
MODEL_TYPE = "logistic"
# ▶ CHANGE ME: drop a feature to see its effect, e.g. remove "maintenance_score"
FEATURES_EXPERIMENT = ["year_built", "sqft", "maintenance_score", "distance_to_transit_km"]
# ▶ CHANGE ME: probability threshold for calling a building "Poor" (default 0.5)
DECISION_THRESHOLD = 0.5

if MODEL_TYPE == "logistic":
    exp_clf = LogisticRegression(max_iter=1000)
else:
    exp_clf = DecisionTreeClassifier(max_depth=4, random_state=42)

exp_clf.fit(X_train[FEATURES_EXPERIMENT], y_train)
exp_proba = exp_clf.predict_proba(X_test[FEATURES_EXPERIMENT])[:, 1]
exp_pred = (exp_proba >= DECISION_THRESHOLD).astype(int)
print(f"Model: {MODEL_TYPE}, threshold: {DECISION_THRESHOLD}")
print(f"Overall accuracy: {accuracy_score(y_test, exp_pred):.2%}")

## Artifact: error table by subgroup + fairness note

In [ ]:
fairness_note = (
    f"The classifier's error rate is {subgroup_error.loc['Low-Income', 'error_rate']:.1%} for "
    f"Low-Income tracts versus {subgroup_error.loc['High-Income', 'error_rate']:.1%} for High-Income "
    f"tracts -- a gap of "
    f"{(subgroup_error.loc['Low-Income', 'error_rate'] - subgroup_error.loc['High-Income', 'error_rate']) * 100:.1f} "
    f"percentage points. This mirrors a common real-world pattern: training labels drawn from "
    f"under-resourced code-enforcement records are noisier, so the model learns that noise. A human "
    f"reviewer must treat 'Poor' predictions in under-inspected neighborhoods with extra scrutiny "
    f"before acting on them (e.g., before triggering an inspection, fine, or funding decision)."
)
print(fairness_note)

subgroup_error.to_csv("lab4_subgroup_error_table.csv")
with open("lab4_fairness_note.txt", "w") as f:
    f.write(fairness_note)
print("\nSaved artifacts: lab4_subgroup_error_table.csv, lab4_fairness_note.txt")

## Reflect (answer in your own words — this is part of your mini-task)

1. What did the tool assume?
2. Who is missing from this data?
3. What would change your recommendation?
4. What must a human verify before this is used?
